In [ ]:
import numpy as np
import random
import gzip
import matplotlib.pyplot as plt
from collections import Counter

def import_bed_file(bed_path):
    fragment_lengths = []
    with gzip.open(bed_path, 'rt') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            cols = line.strip().split('\t')
            col1 = int(cols[1])
            col2 = int(cols[2])
            fragment_lengths.append(col2 - col1)
    return fragment_lengths, Counter(fragment_lengths)

def normalize_frequency(length_freq):
    total = sum(length_freq.values())
    return {length: freq / total for length, freq in length_freq.items()}

def import_reference_histogram(ref_path):
    ref_freq = {}
    with open(ref_path) as f:
        for line in f:
            if not line.strip():
                continue
            length, freq = map(float, line.strip().split('\t'))
            ref_freq[int(length)] = freq
    return ref_freq

def rescale_query_to_reference(fragment_lengths, length_freq, ref_freq_dict, scale_factor=0.3):
    frag_by_length = {}
    for length in fragment_lengths:
        frag_by_length.setdefault(length, []).append(length)
    
    total_fragments = sum(length_freq.values())
    target_total = int(total_fragments * scale_factor)
    rescaled_fragments = []

    for length, ref_freq in ref_freq_dict.items():
        if length in frag_by_length:
            desired_count = int(ref_freq * target_total)
            current_pool = frag_by_length[length]
            sampled = random.sample(current_pool, min(len(current_pool), desired_count))
            rescaled_fragments.extend(sampled)

    rescaled_counts = Counter(rescaled_fragments)
    total_rescaled = sum(rescaled_counts.values())
    rescaled_freq = {i: j / total_rescaled for i, j in rescaled_counts.items()}

    return rescaled_freq, total_fragments, total_rescaled

def plot_distributions(query_freq, ref_freq, rescaled_freq, total_query, total_rescaled):
    query_x = sorted(query_freq.keys())
    query_y = [query_freq[x] for x in query_x]
    
    ref_x = sorted(ref_freq.keys())
    ref_y = [ref_freq[x] for x in ref_x]
    
    rescaled_x = sorted(rescaled_freq.keys())
    rescaled_y = [rescaled_freq[x] for x in rescaled_x]

    plt.figure(figsize=(18, 8))
    plt.plot(query_x, query_y, label=f'Query (n = {total_query:,})', color='blue')
    plt.plot(ref_x, ref_y, label='Reference', color='green')
    plt.plot(rescaled_x, rescaled_y, 'x', label=f'Rescaled (n = {total_rescaled:,})', color='orange', markersize=8, markeredgewidth=2)
    plt.xlabel("Fragment Length (bp)", fontsize=24)
    plt.ylabel("Normalized Frequency [A.U.]", fontsize=24)
    plt.title("Rescaled Fragment Length Distribution", fontsize=24)
    plt.xlim(0, 700)
    plt.ylim(0, 0.025)
    plt.legend(fontsize=14)
    plt.tight_layout()
    plt.savefig("./rescaled_prediction.png")
    plt.show()

if __name__ == "__main__":
    bed_path = './query.bed.gz'
    ref_path = './reference.hist'

    fragment_lengths, length_freq = import_bed_file(bed_path)
    normalized_query_freq = normalize_frequency(length_freq)
    ref_freq_dict = import_reference_histogram(ref_path)

    rescaled_freq, total_query, total_rescaled = rescale_query_to_reference(fragment_lengths, length_freq, ref_freq_dict)

    plot_distributions(normalized_query_freq, ref_freq_dict, rescaled_freq, total_query, total_rescaled)


In [ ]:
import gzip
import numpy as np
import matplotlib.pyplot as plt
import random
import sys
import os
import argparse

def read_reference_hist(hist_file):
    """Read the reference histogram file and return bin indices and frequencies."""
    bins = []
    frequencies = []
    
    with open(hist_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                try:
                    bin_idx = int(parts[0])
                    frequency = float(parts[1])
                    bins.append(bin_idx)
                    frequencies.append(frequency)
                except ValueError:
                    continue
    
    return np.array(bins), np.array(frequencies)

def read_query_bed(bed_file):
    """Read the query bed file and extract values for binning."""
    values = []
    
    if bed_file.endswith('.gz'):
        opener = gzip.open
        mode = 'rt'  
    else:
        opener = open
        mode = 'r'
    
    with opener(bed_file, mode) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:  
                start = int(parts[1])
                end = int(parts[2])
                length = end - start
                values.append(length)
    
    return np.array(values)

def bin_values(values, min_bin=24, max_bin=700):
    """Bin the values and calculate normalized frequencies."""
    bins = range(min_bin, max_bin+1)
    hist, bin_edges = np.histogram(values, bins=bins)
    
    total = np.sum(hist)
    if total > 0:
        normalized_hist = hist / total
    else:
        normalized_hist = hist
    
    bin_indices = np.arange(min_bin, max_bin)
    
    return bin_indices, normalized_hist

def calculate_scaling_factors(ref_freq, query_freq):
    """Calculate scaling factors for each bin to match reference distribution."""
    scaling_factors = np.zeros_like(ref_freq)
    mask = query_freq > 0
    scaling_factors[mask] = ref_freq[mask] / query_freq[mask]
    
    window_size = 5
    scaling_factors = np.convolve(scaling_factors, np.ones(window_size)/window_size, mode='same')
    
    max_factor = 10.0
    scaling_factors = np.clip(scaling_factors, 0, max_factor)
    
    return scaling_factors

def subsample_query_data(query_values, bin_indices, scaling_factors, min_bin=24, max_bin=700, target_count=4295020):
    """Subsample the query data based on scaling factors with a target count."""
    subsampled_values = []
    original_entries = []
    
    bin_to_factor = {min_bin + i: factor for i, factor in enumerate(scaling_factors)}
    
    current_scaling_sum = 0
    bin_counts = {}
    
    for value in query_values:
        if min_bin <= value < max_bin:
            bin_idx = value - min_bin
            if bin_idx < len(scaling_factors):
                bin_counts[bin_idx] = bin_counts.get(bin_idx, 0) + 1
                
    for bin_idx, count in bin_counts.items():
        if bin_idx < len(scaling_factors):
            current_scaling_sum += count * scaling_factors[bin_idx]
    
    adjustment_factor = target_count / current_scaling_sum if current_scaling_sum > 0 else 1.0
    
    for i, value in enumerate(query_values):
        if min_bin <= value < max_bin:
            bin_idx = value - min_bin
            if bin_idx < len(scaling_factors):
                final_scaling = scaling_factors[bin_idx] * adjustment_factor
                if random.random() < final_scaling:
                    subsampled_values.append(value)
                    original_entries.append(i)
    
    return np.array(subsampled_values), np.array(original_entries)

def write_subsampled_bed(original_bed_file, original_indices, output_file):
    """Write a new BED file with subsampled entries based on original indices."""
    if original_bed_file.endswith('.gz'):
        opener = gzip.open
        read_mode = 'rt'  
    else:
        opener = open
        read_mode = 'r'
    
    indices_set = set(original_indices)
    
    if output_file.endswith('.gz'):
        write_opener = gzip.open
        write_mode = 'wt'  
    else:
        write_opener = open
        write_mode = 'w'
    
    with opener(original_bed_file, read_mode) as infile, write_opener(output_file, write_mode) as outfile:
        for i, line in enumerate(infile):
            if i in indices_set:
                outfile.write(line)
                
def plot_distributions(ref_bins, ref_freq, query_bins, query_freq, subsampled_bins, subsampled_freq, output_file):
    """Plot the reference, original query, and subsampled query distributions."""
    plt.figure(figsize=(12, 8))
    
    plt.rcParams['axes.linewidth'] = 1.0
    plt.rcParams['axes.edgecolor'] = 'black'
    plt.rcParams['xtick.major.width'] = 1.0
    plt.rcParams['ytick.major.width'] = 1.0
    
    query_count = len(query_values)
    plt.plot(query_bins, query_freq, '-', color='#8800cc', linewidth=1.5, label=f'Query (n = {query_count:,})')
    
    plt.plot(ref_bins, ref_freq, '-', color='#00aa00', linewidth=1.5, label='Reference')
    
    subsampled_count = len(subsampled_values)
    plt.plot(subsampled_bins, subsampled_freq, 'cyan', marker='*', 
             markersize=6, markevery=1, linewidth=1.0, label=f'Rep 0 (n = {subsampled_count:,})')
    
    plt.xlim(0, 700)
    plt.ylim(0, 0.025)
    
    plt.xticks(np.arange(0, 701, 100))
    plt.yticks(np.arange(0, 0.026, 0.005))
    
    plt.xlabel('Bin')
    plt.ylabel('Normalized Frequency')
    
    plt.legend(loc='upper right', frameon=False)
    
    plt.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    
    plt.show()
    
    def parse_arguments():
    parser = argparse.ArgumentParser(description='Rescale query data to match reference distribution.')
    parser.add_argument('--reference', '-r', default='reference.hist',
                        help='Path to reference histogram file (default: reference.hist)')
    parser.add_argument('--query', '-q', default='query.bed.gz',
                        help='Path to query BED file (default: query.bed.gz)')
    parser.add_argument('--output', '-o', default='subsampled_query.bed.gz',
                        help='Path to output subsampled BED file (default: subsampled_query.bed.gz)')
    parser.add_argument('--plot', '-p', default='distribution_comparison.png',
                        help='Path to output plot file (default: distribution_comparison.png)')
    parser.add_argument('--min-bin', type=int, default=24,
                        help='Minimum bin value (default: 24)')
    parser.add_argument('--max-bin', type=int, default=700,
                        help='Maximum bin value (default: 700)')
    return parser.parse_args()

reference_hist_file = 'reference.hist'
query_bed_file = 'query.bed.gz'
output_bed_file = 'subsampled_query.bed.gz'
plot_file = 'distribution_comparison.png'
min_bin = 24
max_bin = 700

if not os.path.isabs(reference_hist_file):
    script_dir = os.getcwd()
    reference_hist_file = os.path.join(script_dir, reference_hist_file)

if not os.path.isabs(query_bed_file):
    script_dir = os.getcwd()
    query_bed_file = os.path.join(script_dir, query_bed_file)

if not os.path.isabs(output_bed_file):
    script_dir = os.getcwd()
    output_bed_file = os.path.join(script_dir, output_bed_file)

if not os.path.isabs(plot_file):
    script_dir = os.getcwd()
    plot_file = os.path.join(script_dir, plot_file)

print(f"Reading reference histogram from: {reference_hist_file}")
ref_bins, ref_freq = read_reference_hist(reference_hist_file)
print(f"Reference histogram has {len(ref_bins)} bins")

print(f"Reading query data from: {query_bed_file}")
query_values = read_query_bed(query_bed_file)
print(f"Read {len(query_values)} entries from query file")

if len(query_values) == 0:
    print("Error: No data read from query file. Please check the file format.")
    raise ValueError("No data in query file")

query_bins, query_freq = bin_values(query_values, min_bin=min_bin, max_bin=max_bin)
print(f"Binned query data into {len(query_bins)} bins")

min_len = min(len(ref_freq), len(query_bins))
ref_freq_trimmed = ref_freq[:min_len]
query_freq_trimmed = query_freq[:min_len]
print(f"Using {min_len} bins for comparison")

scaling_factors = calculate_scaling_factors(ref_freq_trimmed, query_freq_trimmed)
print("Calculated scaling factors for subsampling")

target_count = 4295020  
subsampled_values, original_indices = subsample_query_data(
    query_values, query_bins, scaling_factors, 
    min_bin=min_bin, max_bin=max_bin, 
    target_count=target_count
)
print(f"Subsampled to {len(subsampled_values)} entries ({len(subsampled_values)/len(query_values)*100:.2f}% of original)")

if len(subsampled_values) == 0:
    print("Warning: No entries were selected during subsampling. Check scaling factors.")
    raise ValueError("No entries selected during subsampling")

subsampled_bins, subsampled_freq = bin_values(subsampled_values, min_bin=min_bin, max_bin=max_bin)
write_subsampled_bed(query_bed_file, original_indices, output_bed_file)
print(f"Wrote subsampled data to: {output_bed_file}")

plot_distributions(ref_bins[:min_len], ref_freq_trimmed, 
                  query_bins[:min_len], query_freq_trimmed, 
                  subsampled_bins, subsampled_freq, 
                  plot_file)

print("\nRescaling completed successfully!")
print(f"Original query entries: {len(query_values)}")
print(f"Subsampled query entries: {len(subsampled_values)}")
print(f"Retention rate: {len(subsampled_values)/len(query_values)*100:.2f}%")